In [ ]:
# Cell 1: Install + Clone + Data

import numpy as np
np.int = int; np.float = float

!pip install torch --index-url https://download.pytorch.org/whl/cpu -q
!pip install scipy pandas numba tqdm numpy biopython msgpack scikit-learn PyYAML -q

!rm -rf Enhanced-ECNet
!git clone https://github.com/itozheng-max/Enhanced-ECNet.git
%cd Enhanced-ECNet

!wget -q https://raw.githubusercontent.com/itozheng-max/Enhanced-ECNet/main/ecnet_rrm_data.tar.gz
!tar xzf ecnet_rrm_data.tar.gz

import sys, time, torch
sys.path.insert(0, ".")
print(f"Py {sys.version.split()[0]} | Torch {torch.__version__} | GPU {torch.cuda.is_available()}")

In [ ]:
# Cell 2: Import

from ecnet.ecnet import ECNet
from ecnet.spatial_mask import SpatialMask
print("OK")

In [ ]:
# Cell 3: Config

FASTA     = "data/RRM.fasta"
TRAIN_TSV = "data/RRM_single.tsv"
TEST_TSV  = "data/RRM_double.tsv"
BRAW      = "data/RRM.braw"
DIST_NPY  = "data/distance_matrix.npy"

MASK = {
    "path": DIST_NPY,
    "mode": "divide",
    "kernel": "hill",
    "d0": 8.0,
    "n": 4,
    "epsilon": 0.10,
}

print(f"Train: {TRAIN_TSV}  Test: {TEST_TSV or 'auto'}")
print(f"Mask: {MASK['mode']} | kernel={MASK.get('kernel','sigmoid')}")

In [ ]:
# Cell 4: Train

ecnet = ECNet(
    output_dir="./output",
    train_tsv=TRAIN_TSV, test_tsv=TEST_TSV,
    fasta=FASTA, ccmpred_output=BRAW,
    use_loc_feat=True, use_glob_feat=False,
    split_ratio=[0.9, 0.1] if TEST_TSV else [0.7, 0.1, 0.2],
    n_ensembles=1, d_embed=20, d_model=64, d_h=64,
    batch_size=64, save_log=True,
    spatial_mask=MASK,
)

t0 = time.time()
ecnet.train(epochs=200, patience=100, eval_freq=20, log_freq=20)
test = ecnet.test(mode="ensemble", save_prediction=True)
print(f"\nDone | {time.time()-t0:.0f}s | Test Spearman: {test['metric']['corr']:.6f}")

In [ ]:
# Cell 5: Baseline vs Sigmoid vs Hill(n=2) vs Hill(n=4)

def run_one(name, mask):
    ecnet = ECNet(
        output_dir=f"./output/{name}",
        train_tsv=TRAIN_TSV, test_tsv=TEST_TSV,
        fasta=FASTA, ccmpred_output=BRAW,
        use_loc_feat=True, use_glob_feat=False,
        split_ratio=[0.9, 0.1] if TEST_TSV else [0.7, 0.1, 0.2],
        n_ensembles=1, d_embed=20, d_model=64, d_h=64,
        batch_size=64, save_log=False,
        spatial_mask=mask,
    )
    ecnet.train(epochs=200, patience=100, eval_freq=20, log_freq=200)
    return ecnet.test(mode="ensemble")["metric"]["corr"]

print("Running...\n")

b = run_one("Baseline", None)
print(f"  Baseline    -> {b:.6f}")

sd = run_one("Sigmoid", {"path": DIST_NPY, "mode": "divide", "kernel": "sigmoid", "d0": 8.0, "gamma": 1.5, "epsilon": 0.10})
print(f"  Sigmoid     -> {sd:.6f}")

h2 = run_one("Hill_n2", {"path": DIST_NPY, "mode": "divide", "kernel": "hill", "d0": 8.0, "n": 2, "epsilon": 0.10})
print(f"  Hill(n=2)   -> {h2:.6f}")

h4 = run_one("Hill_n4", {"path": DIST_NPY, "mode": "divide", "kernel": "hill", "d0": 8.0, "n": 4, "epsilon": 0.10})
print(f"  Hill(n=4)   -> {h4:.6f}")

print(f"\n{'='*55}")
print(f"  Baseline     : {b:.6f}")
print(f"  Sigmoid      : {sd:.6f}  ({sd-b:+.4f})")
print(f"  Hill (n=2)   : {h2:.6f}  ({h2-b:+.4f})")
print(f"  Hill (n=4)   : {h4:.6f}  ({h4-b:+.4f})")
print(f"{'='*55}")